# ForestWatch Papua — Training **MODEL: DeepLabV3+**

**Jalankan di KOMPUTER 2.** Arsitektur: `deeplabv3plus` + `resnet50` (ImageNet). Resep training (loss, class weights, weighted sampler, AMP, freeze-encoder warmup, cosine LR) **identik** dengan ke-2 model lain → perbandingan apple-to-apple.

Output ditulis ke **folder khusus model ini** di gdrive: `ForestWatch_Outputs/Model_Comparison/model_2_deeplabv3plus/` (checkpoint, kurva, metrics, confusion, ONNX, summary) → 3 mesin paralel **tidak saling timpa**.

Setelah ke-3 model selesai, jalankan `compare_and_select_best_model.ipynb`.

## Bagian 0 — Setup environment (Colab **atau** Komputer Lab)

Notebook ini **berdiri sendiri**. Ia **memuat hasil EDA & preprocessing**
(Bagian 1–14 dari `forestwatch_papua_full_pipeline.ipynb`) yang sudah tersimpan
di Google Drive — **tidak menghitung ulang**.

- **Google Colab** → set `ENV = "colab"`. Sel setup meng-clone repo, install
  package, lalu mount Drive.
- **Komputer lab** → set `ENV = "lab"`. Prasyarat **sekali saja**:
  1. Install **Google Drive for Desktop**, login akun yang sama, set folder
     `Satria Data 3.0` ke mode **Mirror** (bukan *Stream-only*) supaya file `.npz`
     benar-benar ada di disk lokal (DataLoader membaca ribuan file tiap epoch).
  2. Di clone repo lokal jalankan: `pip install -e ".[ml]"`.
  3. Sesuaikan `DRIVE_ROOT` ke path mount Drive Desktop (mis. `G:/My Drive/Satria Data 3.0`).

In [ ]:
# === Bagian 0 — Setup (set ENV = "colab" ATAU "lab") ===
ENV = "lab"   # ganti "colab" kalau jalan di Google Colab
from pathlib import Path

if ENV == "colab":
    import subprocess, sys, importlib
    # clone pertama kali / pull sesi berikutnya (selalu kode terbaru), lalu install.
    subprocess.run(
        "cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
        "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
        shell=True, check=False,
    )
    subprocess.run("pip install -q -e /content/fw_repo[gee,gis,ml]", shell=True, check=False)
    if "/content/fw_repo/src" not in sys.path:
        sys.path.insert(0, "/content/fw_repo/src")
    for _m in [m for m in list(sys.modules) if m == "forestwatch" or m.startswith("forestwatch.")]:
        del sys.modules[_m]
    importlib.invalidate_caches()
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Satria Data 3.0")
elif ENV == "lab":
    DRIVE_ROOT = Path(r"G:/My Drive/Satria Data 3.0")   # <- SESUAIKAN path mount Drive Desktop lab
else:
    raise ValueError("ENV harus 'colab' atau 'lab'")

assert DRIVE_ROOT.exists(), (
    f"DRIVE_ROOT {DRIVE_ROOT} tidak ada — cek mount Drive / sync (mode Mirror)."
)

import torch
_gpu = f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""
print(f"ENV={ENV} | DRIVE_ROOT={DRIVE_ROOT} | CUDA={torch.cuda.is_available()}{_gpu}")

In [ ]:
# === Deklarasi path gdrive (hasil EDA & preprocessing tersimpan di sini) ===
TILES_T1   = DRIVE_ROOT / 'ForestWatch_Tiles_T1'
TILES_T2   = DRIVE_ROOT / 'ForestWatch_Tiles_T2'
PATCH_DIR  = DRIVE_ROOT / 'ForestWatch_Patches'           # patch Papua (+ ckpt kanonik)
PATCHES_TRANSFER  = DRIVE_ROOT / 'ForestWatch_Patches_Transfer'
AUGMENTED_PATCHES = DRIVE_ROOT / 'Augmented_Patches'
DIST_DIR   = DRIVE_ROOT / 'Distribution_Reports'
MASK_DIR   = DRIVE_ROOT / 'ForestWatch_Masks'
OUT_DIR    = DRIVE_ROOT / 'ForestWatch_Outputs'
MODELS_ROOT = OUT_DIR / 'Model_Comparison'                # folder induk 3 model
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

from forestwatch.config import load_config
from forestwatch.utils.io import save_json, load_json
cfg = load_config()
print('Resep training :', cfg['training']['loss']['type'],
      '| epochs:', cfg['training']['epochs'], '| batch:', cfg['training']['batch_size'])
print('MODELS_ROOT    :', MODELS_ROOT)

In [ ]:
# === Muat hasil PREPROCESSING dari gdrive (deklarasi + panggil; BUKAN hitung ulang) ===
# Split sumber-aware IDENTIK Bagian 14.1 (seed=42): val/test = Papua-only holdout;
# transfer + augmentasi offline -> train. Class weights dari distribusi Bagian 14.3.
import json
from forestwatch.constants import N_CLASSES, CLASS_NAMES
from forestwatch.data import build_dataloaders_from_files, list_patches, split_files
from forestwatch.training.metrics import median_frequency_weights

papua_files    = list_patches(PATCH_DIR)
transfer_files = list_patches(PATCHES_TRANSFER)
aug_files      = list_patches(AUGMENTED_PATCHES)
assert papua_files, (
    f"Tidak ada patch di {PATCH_DIR}. Pastikan Bagian 1-14 (notebook utama) sudah "
    "dijalankan & Google Drive sudah selesai sync."
)

train_p, val_p, test_p = split_files(papua_files, train_ratio=0.8, val_ratio=0.1, seed=42)
final_train_files = list(train_p) + list(transfer_files) + list(aug_files)
print(f"train={len(final_train_files)} (papua={len(train_p)}+transfer={len(transfer_files)}"
      f"+aug={len(aug_files)}), val={len(val_p)}, test={len(test_p)} (Papua holdout)")

_post_aug = DIST_DIR / 'distribution_post_augmentation_on_target.json'
assert _post_aug.exists(), (
    f"{_post_aug} belum ada — jalankan Bagian 14.3 di notebook utama dulu "
    "(recompute distribusi TRAIN FINAL + median-frequency weights)."
)
dist_final = {int(k): int(v) for k, v in json.load(open(_post_aug))['counts'].items()}
class_weights = median_frequency_weights(dist_final, n_classes=N_CLASSES)
print('class_weights (median-freq):', [round(float(w), 3) for w in class_weights])

In [ ]:
# === Bundling .tar dataset (anti-bottleneck I/O FUSE/rclone) -> extract ke disk lokal ===
# Sekali dibuat (SHARED lintas 3 notebook model + notebook banding, idempoten -> notebook
# lain tinggal extract), lalu tiap sesi training extract cepat ke disk lokal. Hanya
# final_train_files/val_p/test_p yang dialihkan ke path lokal (set & isi file IDENTIK,
# urutan boleh beda -- tidak masalah utk WeightedRandomSampler/DataLoader shuffle=False).
from forestwatch.data.dataset import create_dataset_archives, extract_dataset_archives

BAHAN_DIR = DRIVE_ROOT / 'Bahan_Training_Model'
LOCAL_DATA_DIR = Path('/content/dataset_local') if ENV == 'colab' else Path.home() / 'dataset_local'


def _items(files, root, prefix):
    return [(f"{prefix}/{Path(f).relative_to(Path(root)).as_posix()}", f) for f in files]


splits = {
    'train': (_items(train_p, PATCH_DIR, 'papua')
              + _items(transfer_files, PATCHES_TRANSFER, 'transfer')
              + _items(aug_files, AUGMENTED_PATCHES, 'aug')),
    'val': _items(val_p, PATCH_DIR, 'papua'),
    'test': _items(test_p, PATCH_DIR, 'papua'),
}
create_dataset_archives(splits, BAHAN_DIR, n_train_parts=7)
local_dirs = extract_dataset_archives(BAHAN_DIR, LOCAL_DATA_DIR)

final_train_files = list_patches(local_dirs['train'])
val_p = list_patches(local_dirs['val'])
test_p = list_patches(local_dirs['test'])
print(f"[lokal] train={len(final_train_files)} val={len(val_p)} test={len(test_p)} -> {LOCAL_DATA_DIR}")


In [ ]:
# === EDA cepat: distribusi piksel -- TRAIN FINAL (papua+transfer+aug) vs Val/Test holdout ===
# Sanity-check sebelum training: TRAIN di sini = final_train_files (train_p + transfer_files
# + aug_files), yaitu data yg BENAR-BENAR dipakai model (sudah termasuk augmentasi offline
# utk kelas minor). Val/Test = Papua-only holdout (val_p/test_p), tanpa augmentasi.
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm


def _count_pixels(files, max_workers=64):
    counts = {c: 0 for c in range(N_CLASSES)}
    if not files:
        return counts

    def _read_lab(f):
        return np.load(f)['lab']

    with ThreadPoolExecutor(max_workers=max_workers) as exe:
        for lab in tqdm(exe.map(_read_lab, files), total=len(files), desc='Counting pixels'):
            u, cnt = np.unique(lab, return_counts=True)
            for cls, n in zip(u.tolist(), cnt.tolist()):
                if 0 <= int(cls) < N_CLASSES:
                    counts[int(cls)] += int(n)
    return counts


split_dist = {
    f'Train FINAL ({len(final_train_files)}p)': _count_pixels(final_train_files),
    f'Val ({len(val_p)}p)': _count_pixels(val_p),
    f'Test ({len(test_p)}p)': _count_pixels(test_p),
}

print('Proporsi piksel per kelas -- Train FINAL (papua+transfer+aug) vs Val/Test holdout:\n')
print(f"  {'Kelas':<16}" + ''.join(f'{name:>22}' for name in split_dist))
for c in range(N_CLASSES):
    row = f"  {CLASS_NAMES[c]:<16}"
    for d in split_dist.values():
        total = sum(d.values()) or 1
        row += f'{100 * d[c] / total:5.1f}% ({d[c]:>10,})'.rjust(22)
    print(row)
print(f"  {'TOTAL piksel':<16}" + ''.join(f'{sum(d.values()):>22,}' for d in split_dist.values()))

# Grafik proporsi per-kelas per-split
import matplotlib.pyplot as plt
x = np.arange(N_CLASSES)
width = 0.25
fig, ax = plt.subplots(figsize=(11, 5))
for i, (name, d) in enumerate(split_dist.items()):
    total = sum(d.values()) or 1
    pct = [100 * d[c] / total for c in range(N_CLASSES)]
    ax.bar(x + (i - 1) * width, pct, width, label=name)
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_ylabel('% piksel dalam split'); ax.set_title('Proporsi Piksel per Kelas -- Train FINAL vs Val/Test (Papua)')
ax.legend(); ax.grid(alpha=0.3, axis='y')
fig.tight_layout(); plt.show()

In [ ]:
# === Worker tuning (dinamis: optimal di lab multi-core, minimal 4 di Colab) ===
import os
N_WORKERS = max(4, min(8, (os.cpu_count() or 2) - 1))
print(f"os.cpu_count()={os.cpu_count()} -> num_workers={N_WORKERS}; "
      "persistent_workers=True, pin_memory=True (di-set saat build DataLoader).")

## Bagian 15 — Build Model + Training (DeepLabV3+)

In [ ]:
# === Identitas model (HANYA cell ini yang beda antar-3 notebook training) ===
MODEL_KEY  = "model_2_deeplabv3plus"
MODEL_ARCH = dict(architecture="deeplabv3plus", encoder_name="resnet50")
MODEL_DIR  = MODELS_ROOT / MODEL_KEY
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH  = MODEL_DIR / 'best_model.pt'      # per-model -> tak bentrok antar mesin
print('MODEL_KEY :', MODEL_KEY, '|', MODEL_ARCH)
print('MODEL_DIR :', MODEL_DIR)

In [ ]:
# === Build DataLoaders + model + loss ===
from forestwatch.model.architecture import build_unet, count_parameters
from forestwatch.model.losses import make_loss_fn

use_sampler = cfg['training'].get('use_weighted_sampler', True)
train_loader, val_loader, test_loader = build_dataloaders_from_files(
    final_train_files, val_p, test_p,
    batch_size=cfg['training']['batch_size'],
    num_workers=N_WORKERS,
    augment_p=cfg['training']['augmentation'],
    class_weights=class_weights if use_sampler else None,
    sampler_cache=MODELS_ROOT / 'patch_sampler_weights_shared.json',  # shared lintas 3 model -> hitung sekali
    persistent_workers=True,
)
print(f"Train {len(train_loader.dataset)} | Val {len(val_loader.dataset)} | Test {len(test_loader.dataset)}")

model = build_unet(
    in_channels=cfg['model']['in_channels'], classes=cfg['model']['classes'],
    encoder_weights=cfg['model']['encoder_weights'], **MODEL_ARCH,
)
print(f"{MODEL_KEY}: {count_parameters(model):,} param trainable")

lc = cfg['training']['loss']
loss_fn = make_loss_fn(
    loss_type=lc['type'],
    class_weights=class_weights if cfg['training'].get('use_class_weights', True) else None,
    tversky_alpha=lc.get('tversky_alpha', 0.3), tversky_beta=lc.get('tversky_beta', 0.7),
    focal_gamma=lc.get('focal_gamma', 2.0),
)
print('Loss:', lc['type'])

In [ ]:
# === Training (transfer-learning 2-tahap + resume + grad-clip) ===
# RE-RUNNABLE: jalankan ulang -> lanjut dari checkpoint per-model bila sesi mati.
import time
from forestwatch.training.trainer import TrainConfig, train

tcfg = TrainConfig(
    epochs=cfg['training']['epochs'], patience=cfg['training']['patience'],
    learning_rate=cfg['training']['learning_rate'], weight_decay=cfg['training']['weight_decay'],
    amp=cfg['training']['amp'], warmup_epochs=cfg['training'].get('warmup_epochs', 3),
    freeze_encoder_epochs=cfg['training'].get('freeze_encoder_epochs', 3),
    grad_clip=cfg['training'].get('grad_clip', 1.0), resume=cfg['training'].get('resume', True),
    seed=cfg['project']['seed'], ckpt_path=CKPT_PATH.as_posix(),
)
_t0 = time.time()
summary = train(model, train_loader, val_loader, loss_fn=loss_fn, cfg=tcfg)
train_minutes = round((time.time() - _t0) / 60, 1)
print(f"\nbest val mIoU = {summary['best_val_iou']:.4f} @ epoch {summary['best_epoch']} | {train_minutes} menit")
print("ckpt   :", summary['ckpt_path'])
print("resume :", summary['resume_path'], " (hapus file ini utk latih dari awal)")

In [ ]:
# === Plot history -> MODEL_DIR ===
import matplotlib.pyplot as plt
from forestwatch.constants import CLASS_COLORS

hist = summary['history']; epochs = [h['epoch'] for h in hist]
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].plot(epochs, [h['train_loss'] for h in hist], label='train', lw=2)
axes[0].plot(epochs, [h['val_loss'] for h in hist], label='val', lw=2)
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(epochs, [h['val_miou'] for h in hist], color='green', lw=2)
axes[1].axhline(0.60, color='orange', ls='--', label='min 0.60')
axes[1].axhline(0.75, color='red', ls='--', label='ideal 0.75')
axes[1].set_title('Val mIoU (makro)'); axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(alpha=0.3)
if hist and 'val_iou_per_class' in hist[-1]:
    for c in range(N_CLASSES):
        ys = [h.get('val_iou_per_class', [float('nan')] * N_CLASSES)[c] for h in hist]
        axes[2].plot(epochs, ys, color=CLASS_COLORS[c], label=CLASS_NAMES[c], lw=1.6)
    axes[2].set_title('Val IoU per-kelas'); axes[2].set_ylim(0, 1)
    axes[2].legend(fontsize=7, ncol=2); axes[2].grid(alpha=0.3)
else:
    axes[2].plot(epochs, [h['lr'] for h in hist], color='purple'); axes[2].set_title('LR')
fig.suptitle(MODEL_KEY); fig.tight_layout()
fig.savefig(MODEL_DIR / 'training_curve.png', dpi=120, bbox_inches='tight'); plt.show()
save_json({'best_val_iou': summary.get('best_val_iou'), 'best_epoch': summary.get('best_epoch'),
           'history': hist}, MODEL_DIR / 'training_history.json')
print('Disimpan:', MODEL_DIR / 'training_curve.png', '+ training_history.json')

In [ ]:
# === Evaluasi test (Papua holdout) -> metrics.json + confusion_matrix.png di MODEL_DIR ===
import numpy as np, torch
import matplotlib.pyplot as plt
from forestwatch.training.trainer import evaluate
from forestwatch.training.metrics import compute_confusion_matrix, metric_summary

model.load_state_dict(torch.load(CKPT_PATH, map_location='cpu'))
preds, targets = evaluate(model, test_loader)
cm = compute_confusion_matrix(preds, targets, n_classes=N_CLASSES)
metrics = metric_summary(cm, class_names=CLASS_NAMES)
print(f"OA={metrics['overall_accuracy']*100:.2f}% | mIoU={metrics['mean_iou']:.4f} | kappa={metrics['kappa']:.4f}")
for row in metrics['per_class']:
    print(f"  {row['class']:<16} IoU={row['iou']:.4f}  F1={row['f1']:.4f}")
save_json(metrics, MODEL_DIR / 'metrics.json')

cm_np = np.array(metrics['confusion_matrix']); cm_norm = cm_np / cm_np.sum(axis=1, keepdims=True).clip(1)
fig, ax = plt.subplots(figsize=(8, 6)); im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if cm_norm[i, j] > 0.5 else 'black')
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'Confusion — {MODEL_KEY}')
fig.colorbar(im, ax=ax); fig.tight_layout()
fig.savefig(MODEL_DIR / 'confusion_matrix.png', dpi=120, bbox_inches='tight'); plt.show()
print('Disimpan:', MODEL_DIR / 'metrics.json', '+ confusion_matrix.png')

In [ ]:
# === Ekspor ONNX + tulis summary.json (dibaca notebook perbandingan) ===
from forestwatch.model.architecture import export_to_onnx
export_to_onnx(model, MODEL_DIR / 'model.onnx', in_channels=cfg['model']['in_channels'],
               patch_size=cfg['inference']['patch_size'], opset_version=13)

save_json({
    'model_key': MODEL_KEY, **MODEL_ARCH,
    'best_val_iou': summary['best_val_iou'], 'best_epoch': summary['best_epoch'],
    'test_mean_iou': metrics['mean_iou'], 'test_overall_accuracy': metrics['overall_accuracy'],
    'test_kappa': metrics['kappa'], 'per_class': metrics['per_class'],
    'n_parameters': count_parameters(model), 'train_minutes': train_minutes,
    'epochs_cfg': cfg['training']['epochs'], 'batch_size': cfg['training']['batch_size'],
}, MODEL_DIR / 'summary.json')
print(f"OK {MODEL_KEY} SELESAI. Semua artefak di: {MODEL_DIR}")
print("   -> setelah ke-3 model selesai, jalankan compare_and_select_best_model.ipynb")